In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

DATA_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/akilli-tarim-karar-sistemi/ml_service/data/"
MODEL_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/akilli-tarim-karar-sistemi/ml_service/models/"

# Veriyi yukle
df = pd.read_csv(DATA_PATH + "normalize_veri.csv")

# Ozellikler (nem eklendi)
ozellikler = ["max_sicaklik", "min_sicaklik", "yagis", "ruzgar_hizi", "nem"]
X = df[ozellikler]
y = df["etiket_kod"]

# Egitim ve test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Modeli egit
model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)
model_rf.fit(X_train, y_train)

# Performans
le = joblib.load(MODEL_PATH + "label_encoder.pkl")
y_pred = model_rf.predict(X_test)

print("=" * 40)
print("RANDOM FOREST PERFORMANSI")
print("=" * 40)
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Ozellik onemleri
ozellik_onemleri = pd.DataFrame({
    "ozellik": ozellikler,
    "onem"   : model_rf.feature_importances_
}).sort_values("onem", ascending=False)

print("Ozellik Onemleri:")
print(ozellik_onemleri)

# Modeli kaydet
joblib.dump(model_rf, MODEL_PATH + "random_forest.pkl")
print("\nModel kaydedildi!")

RANDOM FOREST PERFORMANSI
              precision    recall  f1-score   support

      riskli       1.00      1.00      1.00      1988
       uygun       1.00      1.00      1.00      1868
 uygun_degil       1.00      1.00      1.00       528

    accuracy                           1.00      4384
   macro avg       1.00      1.00      1.00      4384
weighted avg       1.00      1.00      1.00      4384

Ozellik Onemleri:
        ozellik      onem
4           nem  0.662773
0  max_sicaklik  0.198163
1  min_sicaklik  0.115466
2         yagis  0.017545
3   ruzgar_hizi  0.006052

Model kaydedildi!
